In [1]:
import anndata as ad
import pandas as pd
import scipy.sparse

def save_expression(adata, output_file, sample_type):
    """
    将 AnnData 对象转换为基因×样本的 tsv 表达矩阵文件。
    
    Parameters:
        adata: AnnData 对象
        output_file: 输出文件名（例如 'sc_expression.tsv'）
        sample_type: 'sc' 或 'st'，仅用于提示
    """
    # 优先使用原始计数（raw），否则使用 .X
    if adata.raw is not None:
        X = adata.raw.X
        var_names = adata.raw.var_names
        print(f"使用 .raw.X 作为 {sample_type} 表达矩阵")
    else:
        X = adata.X
        var_names = adata.var_names
        print(f"使用 .X 作为 {sample_type} 表达矩阵（注意：可能不是原始计数）")
    
    # 如果是稀疏矩阵，转为稠密（如果内存允许）
    if scipy.sparse.issparse(X):
        X = X.toarray()
    
    # 转置：基因 × 样本（细胞/spot）
    df = pd.DataFrame(X.T, index=var_names, columns=adata.obs_names)
    
    # 保存为 tsv，第一列为基因名（表头为 'Gene'）
    df.to_csv(output_file, sep='\t', index=True, index_label='Gene')
    print(f"已保存 {sample_type} 表达文件到 {output_file}，形状: {df.shape}")

# ----- 主程序 -----
sc_data_path = "/home/qyyuan/project/ST_CP/JE/method/data/scRNA_subsampled_20k.h5ad"
st_data_path = "/home/qyyuan/project/ST_CP/JE/compare/groundtruth/NM_yangjy_X/data/Visium_FAD.h5ad"


sc_data_path = "/home/qyyuan/project/ST_CP/JE/simulation/MERFISH/adata_sc_mouse1sample1_mouse1_slice50_celltype.h5ad"
st_data_path = "/home/qyyuan/project/ST_CP/JE/simulation/MERFISH/adata_st_mouse1sample1_mouse1_slice50_drop.h5ad"


# 读取 AnnData
sc_adata = ad.read_h5ad(sc_data_path)
st_adata = ad.read_h5ad(st_data_path)

# 生成两个文件（可自定义输出路径）
save_expression(sc_adata, "sc_expression.tsv", "scRNA-seq")
save_expression(st_adata, "st_expression.tsv", "ST")


使用 .X 作为 scRNA-seq 表达矩阵（注意：可能不是原始计数）
已保存 scRNA-seq 表达文件到 sc_expression.tsv，形状: (254, 4198)
使用 .X 作为 ST 表达矩阵（注意：可能不是原始计数）
已保存 ST 表达文件到 st_expression.tsv，形状: (254, 424)


In [4]:
import anndata as ad
import pandas as pd
import scipy.sparse
import re

def clean_cell_type(label):
    """清理细胞类型字符串，只保留字母、数字、下划线，其他替换为下划线"""
    if not isinstance(label, str):
        label = str(label)
    # 替换除字母数字下划线外的字符为 '_'
    clean = re.sub(r'[^a-zA-Z0-9_]', '_', label)
    # 合并多个连续下划线为单个
    clean = re.sub(r'_+', '_', clean)
    # 去除首尾下划线
    clean = clean.strip('_')
    if clean == '':
        clean = 'unknown'
    return clean



def save_cell_labels(adata, output_file):
    """
    生成细胞类型标签文件：两列，cell_ID 和 cell_type
    """
    labels = adata.obs['celltype'].copy()
    # 清理特殊字符
    labels_clean = labels.apply(clean_cell_type)
    df = pd.DataFrame({
        'Cell IDs': adata.obs_names,
        'CellType': labels_clean
    })
    df.to_csv(output_file, sep='\t', index=False)
    print(f"已保存细胞类型标签到 {output_file}，共 {len(df)} 个细胞")

def save_spatial_coords(adata, output_file):
    """
    生成空间坐标文件：三列，spot_ID, row, col
    """
    coords = adata.obsm['spatial']  # shape (n_spots, 2)
    # 假设第一列为 row，第二列为 col（可根据实际情况交换）
    df = pd.DataFrame({
        'SpotID': adata.obs_names,
        'row': coords[:, 0],
        'col': coords[:, 1]
    })
    df.to_csv(output_file, sep='\t', index=False)
    print(f"已保存空间坐标到 {output_file}，共 {len(df)} 个 spots")

# ----- 主程序 -----
sc_data_path = "/home/qyyuan/project/ST_CP/JE/method/data/scRNA_subsampled_20k.h5ad"
st_data_path = "/home/qyyuan/project/ST_CP/JE/compare/groundtruth/NM_yangjy_X/data/Visium_FAD.h5ad"


sc_data_path = "/home/qyyuan/project/ST_CP/JE/simulation/MERFISH/adata_sc_mouse1sample1_mouse1_slice50_celltype.h5ad"
st_data_path = "/home/qyyuan/project/ST_CP/JE/simulation/MERFISH/adata_st_mouse1sample1_mouse1_slice50_drop.h5ad"

sc_adata = ad.read_h5ad(sc_data_path)
st_adata = ad.read_h5ad(st_data_path)



# 2. 生成细胞类型标签
save_cell_labels(sc_adata, "cell_type_labels.tsv")

# 3. 生成空间坐标
save_spatial_coords(st_adata, "spatial_coords.tsv")

print("\n所有文件生成完毕！")

已保存细胞类型标签到 cell_type_labels.tsv，共 4198 个细胞
已保存空间坐标到 spatial_coords.tsv，共 424 个 spots

所有文件生成完毕！


In [ ]:
%cd /home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/Cytospace/
!cytospace -sp /home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/Cytospace/sc_expression.tsv \
    -ctp /home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/Cytospace/cell_type_labels.tsv \
    -stp /home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/Cytospace/st_expression.tsv \
    -cp /home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/Cytospace/spatial_coords.tsv \
    -o cytospace_results -sm lap_CSPR

SyntaxError: invalid syntax (1563147116.py, line 1)

In [5]:
cytospace -sp sc_expression.tsv -ctp cell_type_labels.tsv -stp st_expression.tsv -cp spatial_coords.tsv -o cytospace_results -sm lap_CSPR

SyntaxError: invalid syntax (3253504744.py, line 1)